Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC WITH tickets_union AS (
# MAGIC     SELECT id, created_at, ticket_group, agent_id
# MAGIC     FROM crm_tickets.tickets
# MAGIC     WHERE (
# MAGIC         date(substr(created_at, 1, 10)) <= date('2025-06-13')
# MAGIC         OR ticket_group IN ('MANUAL_VKYC_AUDIT')
# MAGIC     )
# MAGIC
# MAGIC     UNION ALL
# MAGIC
# MAGIC     SELECT id, created_at, ticket_group, agent_id
# MAGIC     FROM crm_nfs_tickets.tickets
# MAGIC     WHERE date(substr(created_at, 1, 10)) > date('2025-06-13')
# MAGIC ),
# MAGIC
# MAGIC filtered AS (
# MAGIC     SELECT
# MAGIC         date(
# MAGIC             date_parse(
# MAGIC                 substring(replace(t.created_at, 'T', ' '), 1, 19),
# MAGIC                 '%Y-%m-%d %H:%i:%s'
# MAGIC             ) + interval '330' minute
# MAGIC         ) AS call_date
# MAGIC     FROM tickets_union t
# MAGIC     WHERE t.ticket_group IN ('VKYC', 'VKYC_HL')
# MAGIC       AND t.agent_id != '67e4f10d-8fb4-4bec-9900-ffe02555617a'  -- exclude Jeethu Soman
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     call_date AS date,
# MAGIC     COUNT(*) AS total_calls
# MAGIC FROM filtered
# MAGIC WHERE call_date = date(at_timezone(current_timestamp, 'Asia/Kolkata')) - INTERVAL '1' DAY
# MAGIC GROUP BY call_date
# MAGIC """
# MAGIC
# MAGIC df = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1SOoEncvdY3O_qyYnevoBzBJXurqKAjbD-L0dmV447eo/edit?gid=1139213660#gid=1139213660"
worksheet_name = "Connected Call Count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

combined_df = pd.concat([df], ignore_index=True)

# Next available row in column A
worksheet = gs_reader.get_sheet(worksheet_name)
existing_col_a = worksheet.col_values(1)
next_row = len(existing_col_a) + 1 if existing_col_a else 1

gs_reader.set_df_in_sheets(
    worksheet_name,
    combined_df,
    include_headers=False,   # data only, no column headers
    row=next_row,
    col=1,
)